# Notebook 02 — `LEFT JOIN`, `RIGHT JOIN`, `USING`, and `UNION`

**Session:** Database - SQL (2)  
**Module:** 02 · Calculus and Linear Algebra

## Goal
Practice outer joins and set-theoretic operations against `sql_store`:
1. Unilateral row preservation using `LEFT JOIN` and `RIGHT JOIN`.
2. The Anti-Join pattern (`WHERE right_key IS NULL`) to detect zero-activity customers and unsold catalog inventory.
3. Chaining multi-level outer joins (`customers` -> `orders` -> `shippers`) safely.
4. Shorthand equi-joins via the `USING (...)` clause.
5. Set combination with `UNION` (deduplicating) vs. `UNION ALL` (streaming multiset).
6. Customer loyalty tier classification (`Bronze`, `Silver`, `Gold`) via `UNION`.

In [ ]:
import sqlite3
from pathlib import Path

SEED_PATH = Path("../data/sql_store_seed.sql")
conn = sqlite3.connect(":memory:")
conn.executescript(SEED_PATH.read_text(encoding="utf-8"))


def query(sql: str) -> list[dict]:
    """Execute a SQL statement and return rows as a list of dictionaries."""
    cur = conn.execute(sql)
    cols = [desc[0] for desc in cur.description]
    return [dict(zip(cols, row, strict=True)) for row in cur.fetchall()]


def show_rows(rows: list[dict], limit: int | None = None) -> None:
    """Pretty-print query results as an aligned ASCII table."""
    display_rows = rows if limit is None else rows[:limit]
    if not display_rows:
        print("(0 rows)")
        return
    cols = list(display_rows[0].keys())
    widths = {c: max(len(c), *(len(str(r[c])) for r in display_rows)) for c in cols}
    header = " | ".join(f"{c:<{widths[c]}}" for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)
    for r in display_rows:
        print(" | ".join(f"{r[c]!s:<{widths[c]}}" for c in cols))


print("Database connected. Ready for outer join operations.")

## 1. `LEFT JOIN` and the Anti-Join Pattern

A `LEFT JOIN` preserves every row from the left table. If no corresponding record exists in the right table, columns from the right table evaluate to `NULL`. Filtering on `WHERE right_table.key IS NULL` isolates un-matched records (the **anti-join**).

In [ ]:
# Auditing all customers and their orders (including zero-order customers)
left_join_rows = query("""
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    c.points,
    o.order_id,
    o.order_date
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
ORDER BY c.customer_id, o.order_id;
""")
print(f"Total rows in outer join: {len(left_join_rows)}")
show_rows(left_join_rows, limit=8)
assert len(left_join_rows) == 19
assert any(r["customer_id"] == 1 and r["order_id"] is None for r in left_join_rows)

In [ ]:
# Anti-join 1: Identify customers who have never placed an order
zero_order_customers = query("""
SELECT
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    c.phone,
    c.points
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
ORDER BY c.customer_id;
""")
print(f"Customers with 0 orders: {len(zero_order_customers)}")
show_rows(zero_order_customers)
assert len(zero_order_customers) == 5
assert [c["customer_id"] for c in zero_order_customers] == [1, 3, 4, 9, 11]

In [ ]:
# Anti-join 2: Identify catalog products that have never been purchased
unsold_products = query("""
SELECT
    p.product_id,
    p.name,
    p.quantity_in_stock,
    p.unit_price
FROM products p
LEFT JOIN order_items oi
    ON p.product_id = oi.product_id
WHERE oi.order_id IS NULL
ORDER BY p.product_id;
""")
print(f"Unsold catalog products: {len(unsold_products)}")
show_rows(unsold_products)
assert len(unsold_products) == 2
assert [p["product_id"] for p in unsold_products] == [7, 12]

## 2. `RIGHT JOIN` and Symmetrical Equivalence

A `RIGHT JOIN` preserves all rows from the right relation. In modern SQLite (version 3.39+), `RIGHT JOIN` is supported natively. Symmetrically, `A RIGHT JOIN B` is equivalent to `B LEFT JOIN A`.

In [ ]:
# Native RIGHT JOIN (orders RIGHT JOIN customers preserves all customers)
right_rows = query("""
SELECT
    c.customer_id,
    c.first_name,
    o.order_id
FROM orders o
RIGHT JOIN customers c
    ON o.customer_id = c.customer_id
ORDER BY c.customer_id, o.order_id;
""")
print(f"RIGHT JOIN rows: {len(right_rows)}")
show_rows(right_rows, limit=5)
assert len(right_rows) == 19

## 3. Chaining Multiple Outer Joins

When tracing fulfillment from customer to order to carrier, both relationships may be optional. Notice that chained joins must remain `LEFT JOIN`; introducing an `INNER JOIN` downstream would discard customers with null order IDs.

In [ ]:
multi_outer = query("""
SELECT
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    o.order_id,
    o.order_date,
    sh.name AS carrier_name
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
LEFT JOIN shippers sh
    ON o.shipper_id = sh.shipper_id
ORDER BY c.customer_id, o.order_id;
""")
print(f"Multi-outer join rows: {len(multi_outer)}")
show_rows(multi_outer, limit=8)
assert len(multi_outer) == 19
assert any(r["customer_id"] == 2 and r["carrier_name"] == "Speedy Express" for r in multi_outer)
assert any(r["customer_id"] == 1 and r["carrier_name"] is None for r in multi_outer)

## 4. Syntactic Brevity with the `USING (...)` Clause

When the primary key and foreign key attributes share identical names across both schemas, `USING (col)` simplifies join syntax and automatically deduplicates projected columns.

In [ ]:
# Single key USING
using_single = query("""
SELECT c.customer_id, c.first_name, o.order_id
FROM customers c
JOIN orders o USING (customer_id)
ORDER BY c.customer_id, o.order_id;
""")

# Composite key USING
using_composite = query("""
SELECT oi.order_id, oi.product_id, oi.quantity, oin.note
FROM order_items oi
JOIN order_item_notes oin USING (order_id, product_id)
ORDER BY oi.order_id, oi.product_id;
""")
assert len(using_single) == 14
assert len(using_composite) == 8
print(
    f"USING queries executed successfully: single={len(using_single)}, composite={len(using_composite)}"
)

## 5. Set Operations: `UNION` vs. `UNION ALL`

`UNION` combines result sets vertically and performs an $O(N \log N)$ deduplication pass. `UNION ALL` performs an $O(N)$ streaming multiset concatenation, retaining all rows.

In [ ]:
# Overlapping queries to demonstrate deduplication
q_union = query("""
SELECT product_id, name FROM products WHERE unit_price < 3.00
UNION
SELECT product_id, name FROM products WHERE quantity_in_stock > 60;
""")

q_union_all = query("""
SELECT product_id, name FROM products WHERE unit_price < 3.00
UNION ALL
SELECT product_id, name FROM products WHERE quantity_in_stock > 60;
""")
print(f"UNION (distinct): {len(q_union)} rows")
print(f"UNION ALL (all with duplicates): {len(q_union_all)} rows")
assert len(q_union) < len(q_union_all)
assert len(q_union) == 10
assert len(q_union_all) == 13

## 6. Analytical Application: Customer Loyalty Tier Segmentation via `UNION`

Segment all 12 customers into discrete tiers based on loyalty points:
- **Bronze:** Points < 1500
- **Silver:** 1500 <= Points <= 2500
- **Gold:** Points > 2500

In [ ]:
tier_rows = query("""
SELECT
    customer_id,
    first_name || ' ' || last_name AS customer_name,
    points,
    'Bronze' AS loyalty_tier
FROM customers
WHERE points < 1500

UNION

SELECT
    customer_id,
    first_name || ' ' || last_name AS customer_name,
    points,
    'Silver' AS loyalty_tier
FROM customers
WHERE points BETWEEN 1500 AND 2500

UNION

SELECT
    customer_id,
    first_name || ' ' || last_name AS customer_name,
    points,
    'Gold' AS loyalty_tier
FROM customers
WHERE points > 2500

ORDER BY points DESC;
""")
print(f"Categorized customers: {len(tier_rows)}")
show_rows(tier_rows)
assert len(tier_rows) == 12
assert tier_rows[0]["loyalty_tier"] == "Gold"
assert tier_rows[-1]["loyalty_tier"] == "Bronze"